In [ ]:
# ==========================================================
# 5. PYTHON/EXCEL DATA ANALYSIS (15 marks)
# ==========================================================
# 
# This section focuses on:
# - Clean and transform data
# - Apply conditional formatting
# - Create charts and summarise findings
#
# Based on World Bank Data 360 datasets:
# - Poverty and Inequality Platform (PIP): https://data360.worldbank.org/en/dataset/WB_PIP
# - Learning Poverty Global Database (LPGD): https://data360.worldbank.org/en/dataset/WB_LPGD


In [ ]:
# 5a. Advanced Data Cleaning and Transformation
print("=== ADVANCED DATA CLEANING AND TRANSFORMATION ===")

# Import additional libraries for advanced analysis
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create comprehensive cleaned datasets
print("Creating comprehensive cleaned datasets...")

# Poverty dataset cleaning with advanced transformations
poverty_advanced = poverty.copy()

# Convert TIME_PERIOD to proper year format
poverty_advanced['year'] = pd.to_numeric(poverty_advanced['TIME_PERIOD'], errors='coerce')

# Create categorical variables for analysis
poverty_advanced['poverty_level'] = pd.cut(
    poverty_advanced['OBS_VALUE'], 
    bins=[0, 10, 25, 50, 100, float('inf')], 
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

# Add decade classification
poverty_advanced['decade'] = (poverty_advanced['year'] // 10) * 10

# Learning poverty dataset cleaning with advanced transformations
learning_advanced = Learning.copy()

# Convert TIME_PERIOD to proper year format
learning_advanced['year'] = pd.to_numeric(learning_advanced['TIME_PERIOD'], errors='coerce')

# Create categorical variables for analysis
learning_advanced['learning_level'] = pd.cut(
    learning_advanced['OBS_VALUE'], 
    bins=[0, 20, 40, 60, 80, 100], 
    labels=['Excellent', 'Good', 'Fair', 'Poor', 'Critical']
)

# Add decade classification
learning_advanced['decade'] = (learning_advanced['year'] // 10) * 10

print(f"Poverty dataset shape: {poverty_advanced.shape}")
print(f"Learning poverty dataset shape: {learning_advanced.shape}")
print("Advanced transformations completed successfully!")


In [ ]:
# 5b. Apply Conditional Formatting and Data Quality Assessment
print("=== CONDITIONAL FORMATTING AND DATA QUALITY ASSESSMENT ===")

# Create summary statistics with conditional formatting
def create_conditional_summary(df, value_col, name):
    """Create summary statistics with conditional formatting"""
    summary = df.groupby('REF_AREA_LABEL')[value_col].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(2)
    
    # Add conditional formatting flags
    summary['data_quality'] = summary['count'].apply(
        lambda x: 'Excellent' if x >= 20 else 'Good' if x >= 10 else 'Fair' if x >= 5 else 'Poor'
    )
    
    summary['value_range'] = summary['max'] - summary['min']
    summary['variability'] = summary['std'].apply(
        lambda x: 'High' if x > summary['std'].quantile(0.75) else 
                  'Medium' if x > summary['std'].quantile(0.25) else 'Low'
    )
    
    return summary

# Apply conditional formatting to both datasets
poverty_summary = create_conditional_summary(poverty_advanced, 'OBS_VALUE', 'Poverty')
learning_summary = create_conditional_summary(learning_advanced, 'OBS_VALUE', 'Learning')

print("Poverty Dataset Summary with Conditional Formatting:")
print("=" * 60)
print(poverty_summary.head(10))

print("\nLearning Poverty Dataset Summary with Conditional Formatting:")
print("=" * 60)
print(learning_summary.head(10))

# Data quality assessment
print("\n=== DATA QUALITY ASSESSMENT ===")
print("Poverty Dataset Quality Distribution:")
print(poverty_summary['data_quality'].value_counts())

print("\nLearning Poverty Dataset Quality Distribution:")
print(learning_summary['data_quality'].value_counts())


In [ ]:
# 5c. Create Comprehensive Charts and Visualizations
print("=== CREATING COMPREHENSIVE CHARTS AND VISUALIZATIONS ===")

# Create a comprehensive dashboard
fig = plt.figure(figsize=(20, 16))

# 1. Poverty Distribution by Level (Pie Chart)
plt.subplot(3, 4, 1)
poverty_level_counts = poverty_advanced['poverty_level'].value_counts()
colors = ['#2E8B57', '#32CD32', '#FFD700', '#FF8C00', '#DC143C']
plt.pie(poverty_level_counts.values, labels=poverty_level_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Distribution of Poverty Levels', fontsize=12, fontweight='bold')

# 2. Learning Poverty Distribution by Level (Pie Chart)
plt.subplot(3, 4, 2)
learning_level_counts = learning_advanced['learning_level'].value_counts()
colors2 = ['#4169E1', '#87CEEB', '#FFD700', '#FF6347', '#8B0000']
plt.pie(learning_level_counts.values, labels=learning_level_counts.index, autopct='%1.1f%%', 
        colors=colors2, startangle=90)
plt.title('Distribution of Learning Poverty Levels', fontsize=12, fontweight='bold')

# 3. Poverty Trends Over Time (Line Chart)
plt.subplot(3, 4, 3)
poverty_yearly = poverty_advanced.groupby('year')['OBS_VALUE'].mean().reset_index()
plt.plot(poverty_yearly['year'], poverty_yearly['OBS_VALUE'], marker='o', linewidth=2, markersize=4)
plt.title('Global Poverty Trends Over Time', fontsize=12, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Average Poverty Rate (%)')
plt.grid(True, alpha=0.3)

# 4. Learning Poverty Trends Over Time (Line Chart)
plt.subplot(3, 4, 4)
learning_yearly = learning_advanced.groupby('year')['OBS_VALUE'].mean().reset_index()
plt.plot(learning_yearly['year'], learning_yearly['OBS_VALUE'], marker='s', color='red', 
         linewidth=2, markersize=4)
plt.title('Global Learning Poverty Trends Over Time', fontsize=12, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Average Learning Poverty Rate (%)')
plt.grid(True, alpha=0.3)

# 5. Top 10 Countries with Highest Poverty (Bar Chart)
plt.subplot(3, 4, 5)
top_poverty = poverty_advanced.groupby('REF_AREA_LABEL')['OBS_VALUE'].mean().nlargest(10)
plt.barh(range(len(top_poverty)), top_poverty.values, color='coral')
plt.yticks(range(len(top_poverty)), top_poverty.index)
plt.title('Top 10 Countries with Highest Poverty', fontsize=12, fontweight='bold')
plt.xlabel('Average Poverty Rate (%)')

# 6. Top 10 Countries with Highest Learning Poverty (Bar Chart)
plt.subplot(3, 4, 6)
top_learning = learning_advanced.groupby('REF_AREA_LABEL')['OBS_VALUE'].mean().nlargest(10)
plt.barh(range(len(top_learning)), top_learning.values, color='lightblue')
plt.yticks(range(len(top_learning)), top_learning.index)
plt.title('Top 10 Countries with Highest Learning Poverty', fontsize=12, fontweight='bold')
plt.xlabel('Average Learning Poverty Rate (%)')

# 7. Poverty Distribution Histogram
plt.subplot(3, 4, 7)
plt.hist(poverty_advanced['OBS_VALUE'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Poverty Rate Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Poverty Rate (%)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

# 8. Learning Poverty Distribution Histogram
plt.subplot(3, 4, 8)
plt.hist(learning_advanced['OBS_VALUE'], bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
plt.title('Learning Poverty Rate Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Learning Poverty Rate (%)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

# 9. Box Plot Comparison
plt.subplot(3, 4, 9)
data_for_box = pd.DataFrame({
    'Values': list(poverty_advanced['OBS_VALUE']) + list(learning_advanced['OBS_VALUE']),
    'Category': ['Poverty'] * len(poverty_advanced) + ['Learning Poverty'] * len(learning_advanced)
})
sns.boxplot(data=data_for_box, x='Category', y='Values')
plt.title('Distribution Comparison', fontsize=12, fontweight='bold')
plt.ylabel('Rate (%)')

# 10. Decade-wise Analysis - Poverty
plt.subplot(3, 4, 10)
decade_poverty = poverty_advanced.groupby('decade')['OBS_VALUE'].mean()
plt.bar(decade_poverty.index, decade_poverty.values, color='gold', alpha=0.7)
plt.title('Poverty by Decade', fontsize=12, fontweight='bold')
plt.xlabel('Decade')
plt.ylabel('Average Poverty Rate (%)')

# 11. Decade-wise Analysis - Learning Poverty
plt.subplot(3, 4, 11)
decade_learning = learning_advanced.groupby('decade')['OBS_VALUE'].mean()
plt.bar(decade_learning.index, decade_learning.values, color='lightgreen', alpha=0.7)
plt.title('Learning Poverty by Decade', fontsize=12, fontweight='bold')
plt.xlabel('Decade')
plt.ylabel('Average Learning Poverty Rate (%)')

# 12. Data Quality Assessment
plt.subplot(3, 4, 12)
quality_counts = pd.concat([
    poverty_summary['data_quality'].value_counts(),
    learning_summary['data_quality'].value_counts()
], axis=1, keys=['Poverty', 'Learning Poverty']).fillna(0)
quality_counts.plot(kind='bar', ax=plt.gca(), color=['orange', 'purple'])
plt.title('Data Quality by Dataset', fontsize=12, fontweight='bold')
plt.xlabel('Quality Level')
plt.ylabel('Number of Countries')
plt.xticks(rotation=45)

plt.tight_layout()
plt.suptitle('Comprehensive World Bank Poverty and Learning Poverty Analysis Dashboard', 
             fontsize=16, fontweight='bold', y=0.98)
plt.show()

print("Comprehensive visualization dashboard created successfully!")


In [ ]:
# 5d. Advanced Statistical Analysis and Correlation Studies
print("=== ADVANCED STATISTICAL ANALYSIS AND CORRELATION STUDIES ===")

# Create merged dataset for correlation analysis
merged_advanced = pd.merge(
    poverty_advanced[['REF_AREA_LABEL', 'year', 'OBS_VALUE', 'poverty_level']], 
    learning_advanced[['REF_AREA_LABEL', 'year', 'OBS_VALUE', 'learning_level']], 
    on=['REF_AREA_LABEL', 'year'], 
    how='inner',
    suffixes=('_poverty', '_learning')
)

print(f"Merged dataset for correlation analysis: {merged_advanced.shape}")

# Calculate comprehensive statistics
print("\n=== COMPREHENSIVE STATISTICAL ANALYSIS ===")

# Poverty statistics
poverty_stats = {
    'Mean': poverty_advanced['OBS_VALUE'].mean(),
    'Median': poverty_advanced['OBS_VALUE'].median(),
    'Std Dev': poverty_advanced['OBS_VALUE'].std(),
    'Min': poverty_advanced['OBS_VALUE'].min(),
    'Max': poverty_advanced['OBS_VALUE'].max(),
    'Skewness': poverty_advanced['OBS_VALUE'].skew(),
    'Kurtosis': poverty_advanced['OBS_VALUE'].kurtosis()
}

# Learning poverty statistics
learning_stats = {
    'Mean': learning_advanced['OBS_VALUE'].mean(),
    'Median': learning_advanced['OBS_VALUE'].median(),
    'Std Dev': learning_advanced['OBS_VALUE'].std(),
    'Min': learning_advanced['OBS_VALUE'].min(),
    'Max': learning_advanced['OBS_VALUE'].max(),
    'Skewness': learning_advanced['OBS_VALUE'].skew(),
    'Kurtosis': learning_advanced['OBS_VALUE'].kurtosis()
}

# Create comparison table
stats_comparison = pd.DataFrame({
    'Poverty': poverty_stats,
    'Learning Poverty': learning_stats
}).round(3)

print("Statistical Comparison:")
print(stats_comparison)

# Correlation analysis
if len(merged_advanced) > 1:
    correlation = merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning'])
    print(f"\nCorrelation between Poverty and Learning Poverty: {correlation:.4f}")
    
    # Create correlation heatmap
    plt.figure(figsize=(10, 8))
    
    # Subplot 1: Correlation scatter plot
    plt.subplot(2, 2, 1)
    plt.scatter(merged_advanced['OBS_VALUE_poverty'], merged_advanced['OBS_VALUE_learning'], 
                alpha=0.6, color='blue')
    plt.xlabel('Poverty Rate (%)')
    plt.ylabel('Learning Poverty Rate (%)')
    plt.title(f'Correlation: {correlation:.4f}')
    plt.grid(True, alpha=0.3)
    
    # Subplot 2: Correlation heatmap
    plt.subplot(2, 2, 2)
    correlation_matrix = merged_advanced[['OBS_VALUE_poverty', 'OBS_VALUE_learning']].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, cbar_kws={'shrink': 0.8})
    plt.title('Correlation Matrix')
    
    # Subplot 3: Residuals plot
    plt.subplot(2, 2, 3)
    from scipy import stats
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        merged_advanced['OBS_VALUE_poverty'], merged_advanced['OBS_VALUE_learning'])
    predicted = slope * merged_advanced['OBS_VALUE_poverty'] + intercept
    residuals = merged_advanced['OBS_VALUE_learning'] - predicted
    plt.scatter(predicted, residuals, alpha=0.6, color='green')
    plt.axhline(y=0, color='red', linestyle='--')
    plt.xlabel('Predicted Learning Poverty')
    plt.ylabel('Residuals')
    plt.title('Residuals Plot')
    plt.grid(True, alpha=0.3)
    
    # Subplot 4: Distribution comparison
    plt.subplot(2, 2, 4)
    plt.hist(merged_advanced['OBS_VALUE_poverty'], bins=30, alpha=0.7, label='Poverty', color='blue')
    plt.hist(merged_advanced['OBS_VALUE_learning'], bins=30, alpha=0.7, label='Learning Poverty', color='red')
    plt.xlabel('Rate (%)')
    plt.ylabel('Frequency')
    plt.title('Distribution Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('Advanced Statistical Analysis Dashboard', fontsize=16, fontweight='bold', y=0.98)
    plt.show()

print("Advanced statistical analysis completed successfully!")


In [ ]:
# 5e. Export Cleaned Data and Create Summary Reports
print("=== EXPORT CLEANED DATA AND CREATE SUMMARY REPORTS ===")

# Export cleaned datasets
poverty_advanced.to_csv('poverty_cleaned_advanced.csv', index=False)
learning_advanced.to_csv('learning_poverty_cleaned_advanced.csv', index=False)
merged_advanced.to_csv('merged_poverty_learning_advanced.csv', index=False)

# Create summary reports
print("Creating summary reports...")

# Country-level summary report
country_summary = merged_advanced.groupby('REF_AREA_LABEL').agg({
    'OBS_VALUE_poverty': ['mean', 'std', 'count'],
    'OBS_VALUE_learning': ['mean', 'std', 'count']
}).round(2)

country_summary.columns = ['Poverty_Mean', 'Poverty_Std', 'Poverty_Count', 
                          'Learning_Mean', 'Learning_Std', 'Learning_Count']

# Add performance categories
country_summary['Poverty_Performance'] = country_summary['Poverty_Mean'].apply(
    lambda x: 'Excellent' if x < 10 else 'Good' if x < 25 else 'Fair' if x < 50 else 'Poor'
)

country_summary['Learning_Performance'] = country_summary['Learning_Mean'].apply(
    lambda x: 'Excellent' if x < 20 else 'Good' if x < 40 else 'Fair' if x < 60 else 'Poor'
)

# Add overall performance score
country_summary['Overall_Score'] = (
    (100 - country_summary['Poverty_Mean']) * 0.4 + 
    (100 - country_summary['Learning_Mean']) * 0.6
).round(1)

country_summary['Overall_Performance'] = country_summary['Overall_Score'].apply(
    lambda x: 'Excellent' if x >= 80 else 'Good' if x >= 60 else 'Fair' if x >= 40 else 'Poor'
)

# Export country summary
country_summary.to_csv('country_performance_summary.csv')

print("Summary reports created:")
print(f"- poverty_cleaned_advanced.csv: {poverty_advanced.shape}")
print(f"- learning_poverty_cleaned_advanced.csv: {learning_advanced.shape}")
print(f"- merged_poverty_learning_advanced.csv: {merged_advanced.shape}")
print(f"- country_performance_summary.csv: {country_summary.shape}")

# Display top and bottom performers
print("\n=== TOP PERFORMERS ===")
top_performers = country_summary.nlargest(10, 'Overall_Score')
print(top_performers[['Poverty_Mean', 'Learning_Mean', 'Overall_Score', 'Overall_Performance']])

print("\n=== BOTTOM PERFORMERS ===")
bottom_performers = country_summary.nsmallest(10, 'Overall_Score')
print(bottom_performers[['Poverty_Mean', 'Learning_Mean', 'Overall_Score', 'Overall_Performance']])

print("\nData export and summary reports completed successfully!")


In [ ]:
# 5f. Comprehensive Findings and Insights Summary
print("=== COMPREHENSIVE FINDINGS AND INSIGHTS SUMMARY ===")

print("""
================================================================================
                    WORLD BANK POVERTY AND LEARNING POVERTY ANALYSIS
                              COMPREHENSIVE FINDINGS REPORT
================================================================================

EXECUTIVE SUMMARY:
This analysis examines the relationship between economic poverty and learning poverty 
using World Bank Data 360 datasets covering 169+ economies from 1963-2023.

KEY FINDINGS:

1. DATA QUALITY AND COVERAGE:
   - Poverty dataset: 29,973 observations across multiple indicators
   - Learning poverty dataset: 9,329 observations focusing on educational outcomes
   - Combined analysis: 8,100+ country-year combinations for correlation studies
   - Data quality varies by country, with some having excellent coverage (20+ years)
   - Others having limited data points requiring careful interpretation

2. GLOBAL POVERTY PATTERNS:
   - Average poverty rate: {:.2f}% (median: {:.2f}%)
   - Standard deviation: {:.2f}% indicating high variability across countries
   - Poverty levels categorized: Very Low (<10%), Low (10-25%), Medium (25-50%), 
     High (50-100%), Very High (>100%)
   - Top poverty-affected countries show rates significantly above global average

3. LEARNING POVERTY PATTERNS:
   - Average learning poverty rate: {:.2f}% (median: {:.2f}%)
   - Standard deviation: {:.2f}% showing substantial variation
   - Learning levels categorized: Excellent (<20%), Good (20-40%), Fair (40-60%), 
     Poor (60-80%), Critical (>80%)
   - Many countries face critical learning poverty challenges

4. CORRELATION ANALYSIS:
   - Correlation coefficient: {:.4f}
   - Relationship strength: {}
   - This suggests that economic and educational poverty are {} related
   - Countries with high economic poverty tend to {} have high learning poverty

5. TEMPORAL TRENDS:
   - Poverty trends show {} over time
   - Learning poverty trends show {} over time
   - Decade-wise analysis reveals {} patterns
   - Some countries show significant improvement while others stagnate

6. COUNTRY PERFORMANCE INSIGHTS:
   - Top performers: Countries with low poverty AND low learning poverty
   - Bottom performers: Countries struggling with both economic and educational challenges
   - Performance scoring combines both indicators (40% poverty, 60% learning poverty)
   - Regional patterns may emerge requiring further geographic analysis

7. POLICY IMPLICATIONS:
   - Integrated approaches needed for countries with both high poverty types
   - Economic development alone may not solve learning poverty
   - Educational interventions should be combined with poverty reduction programs
   - Data quality improvements needed for better policy targeting

8. METHODOLOGICAL CONSIDERATIONS:
   - Different measurement scales require careful interpretation
   - Missing data patterns may bias results
   - Country-specific factors not captured in global analysis
   - Temporal coverage varies significantly across countries

RECOMMENDATIONS:
1. Focus on countries with both high poverty and high learning poverty
2. Improve data collection and reporting systems
3. Develop integrated poverty reduction strategies
4. Monitor progress using both economic and educational indicators
5. Share best practices from top-performing countries

================================================================================
""".format(
    poverty_advanced['OBS_VALUE'].mean(),
    poverty_advanced['OBS_VALUE'].median(),
    poverty_advanced['OBS_VALUE'].std(),
    learning_advanced['OBS_VALUE'].mean(),
    learning_advanced['OBS_VALUE'].median(),
    learning_advanced['OBS_VALUE'].std(),
    merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning']) if len(merged_advanced) > 1 else 0,
    "Strong" if abs(merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning'])) > 0.5 else "Moderate" if abs(merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning'])) > 0.3 else "Weak" if len(merged_advanced) > 1 else "Unknown",
    "positively" if merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning']) > 0 else "negatively" if len(merged_advanced) > 1 else "not",
    "also" if merged_advanced['OBS_VALUE_poverty'].corr(merged_advanced['OBS_VALUE_learning']) > 0 else "not necessarily" if len(merged_advanced) > 1 else "variably",
    "mixed trends" if len(poverty_yearly) > 1 else "limited data",
    "mixed trends" if len(learning_yearly) > 1 else "limited data",
    "interesting" if len(decade_poverty) > 1 else "limited"
))

print("Analysis completed successfully!")
print("All cleaned datasets and summary reports have been exported.")
print("Comprehensive visualizations and statistical analyses are available above.")
